In [1]:
import h5netcdf

fn = "./13901616_nc/V_12hmean_coarse.nc"

with h5netcdf.File(fn, "r") as nc:
    print(nc)
    tlen = nc.variables["time"].shape[0]
    vlen = nc.variables["V"].shape[0]
    print("time length:", nc.variables["time"][-1])
    print("data length:", vlen)
    print("equal?", tlen == vlen)

<h5netcdf.File 'V_12hmean_coarse.nc' (mode r)>
Dimensions:
    k: <h5netcdf.Dimension 'k': size 49>
    j_g: <h5netcdf.Dimension 'j_g': size 120>
    i: <h5netcdf.Dimension 'i': size 60>
    time: <h5netcdf.Dimension 'time': size 7411 (unlimited)>
Groups:
Variables:
    time: ('time',) int64
    V: ('time', 'k', 'j_g', 'i') float32
Attributes:
    origin_iter: 13901616
    hourly_step: 144
    block_size: 12
    time_is_block_start: 1
    box_j: 10
    box_i: 10
    varname: 'V'
time length: 26706096
data length: 7411
equal? True


In [1]:
import os
import numpy as np
import xarray as xr
import h5netcdf
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"
import read_mitgcm

from coarsen_UVWTS import coarse_field


def make_mitgcm_loader_onevar(grid_dir, read_mitgcm, var: str, dims):
    """
    Returns load_snapshot(it) -> xr.DataArray (3D) for ONE variable.
    This is faster than loading a full Dataset each time.
    """
    def load_snapshot(it: int) -> xr.DataArray:
        da = read_mitgcm._ocean_var(grid_dir, var, it, dims)
        da = da.drop_vars("time", errors="ignore")
        da.name = var
        return da
    return load_snapshot


def _12h_mean_to_netcdf_onevar(
    start_time_index: int,
    end_time_index: int,
    load_snapshot,                   # fn(it:int) -> xr.DataArray (3D)
    out_nc: str,                     # e.g. "./out_nc/T_12hmean_coarse.nc"
    ds_fine: xr.Dataset,
    box_j: int,
    box_i: int,
    varname: str,                    # e.g. "T"
    block_size: int = 12,
    step: int = 144,
    time_name: str = "time",
    dtype=np.float32,
    skip_missing: bool = False,
    print_every: int = 20,
):
    start_time_index = int(start_time_index)
    end_time_index   = int(end_time_index)
    step             = int(step)

    if end_time_index < start_time_index:
        raise ValueError("end_time_index must be >= start_time_index")

    out_dir = os.path.dirname(out_nc) or "."
    os.makedirs(out_dir, exist_ok=True)

    block_span = block_size * step

    # ---- read existing file info (saved block starts + origin alignment)
    saved = set()
    origin_iter = start_time_index

    if os.path.exists(out_nc):
        with h5netcdf.File(out_nc, "r") as nc:
            if time_name in nc.variables:
                saved = set(map(int, nc.variables[time_name][:].tolist()))
            if "origin_iter" in nc.attrs:
                origin_iter = int(nc.attrs["origin_iter"])

            # optional safety: ensure the file matches this variable/coarsen setup
            if "varname" in nc.attrs and str(nc.attrs["varname"]) != str(varname):
                raise ValueError(f"{out_nc} already stores varname={nc.attrs['varname']} but you requested {varname}")
            if "box_j" in nc.attrs and int(nc.attrs["box_j"]) != int(box_j):
                raise ValueError("box_j mismatch with existing file")
            if "box_i" in nc.attrs and int(nc.attrs["box_i"]) != int(box_i):
                raise ValueError("box_i mismatch with existing file")

    # last hour of block must be <= end_time_index
    max_n = int(np.floor((end_time_index - origin_iter - (block_size - 1) * step) / block_span))
    if max_n < 0:
        print("Not enough hourly data to form even one full 12-hour block yet.")
        return

    # ---- init file (for ONE variable only)
    def _init_file(coarse_da: xr.DataArray):
        # coarse_da is 3D (no time dim)
        spatial_dims = tuple(coarse_da.dims)
        spatial_sizes = {d: int(coarse_da.sizes[d]) for d in spatial_dims}

        with h5netcdf.File(out_nc, "w") as nc:
            # dimensions
            nc.dimensions[time_name] = None
            for d, n in spatial_sizes.items():
                nc.dimensions[d] = n

            # attrs
            nc.attrs["origin_iter"] = int(origin_iter)
            nc.attrs["hourly_step"] = int(step)
            nc.attrs["block_size"] = int(block_size)
            nc.attrs["time_is_block_start"] = 1
            nc.attrs["box_j"] = int(box_j)
            nc.attrs["box_i"] = int(box_i)
            nc.attrs["varname"] = str(varname)

            # time coord
            tvar = nc.create_variable(
                time_name,
                dimensions=(time_name,),
                dtype="i8",
                chunks=(1,),
                maxshape=(None,),
            )
            tvar.attrs["long_name"] = "start index of each 12-hour averaging block"
            tvar.attrs["units"] = "model_iteration_index"

            # data var: (time, spatial...)
            dims = (time_name,) + spatial_dims
            chunks = (1,) + tuple(spatial_sizes[d] for d in spatial_dims)
            maxshape = (None,) + tuple(spatial_sizes[d] for d in spatial_dims)

            v = nc.create_variable(
                varname,
                dimensions=dims,
                dtype="f4",
                chunks=chunks,
                maxshape=maxshape,
            )
            v.attrs["long_name"] = varname

    appended = 0
    skipped_saved = 0
    skipped_missing = 0

    nc_app = None
    try:
        for n in range(max_n + 1):
            s = origin_iter + n * block_span

            if print_every and (n % print_every == 0):
                print(f"[try] n={n}/{max_n}  block_start={s}", flush=True)

            if s in saved:
                skipped_saved += 1
                continue

            block_iters = [s + m * step for m in range(block_size)]

            # ---- fast 12h mean for ONE var (sum numpy arrays)
            snap0 = None
            sum_arr = None

            try:
                for it in block_iters:
                    da = load_snapshot(int(it))  # xr.DataArray (3D)

                    if snap0 is None:
                        snap0 = da
                        sum_arr = da.values.astype(dtype, copy=True)
                    else:
                        sum_arr += da.values.astype(dtype, copy=False)

            except Exception as e:
                if skip_missing:
                    skipped_missing += 1
                    print(f"  -> missing/unreadable snapshot in block_start={s}, skip ({e})", flush=True)
                    continue
                raise

            mean_arr = sum_arr / block_size

            # rebuild mean DataArray with same coords/dims as first snapshot
            mean_da = snap0.copy(data=mean_arr)
            mean_da.name = varname

            # ---- coarsen AFTER averaging
            coarse_da = coarse_field(varname, mean_da, ds_fine, box_j=box_j, box_i=box_i)
            coarse_da = coarse_da.astype("float32")
            coarse_da.name = varname

            # ---- initialize file once we know coarse shape
            if not os.path.exists(out_nc):
                _init_file(coarse_da)

            # ---- open once for appending
            if nc_app is None:
                nc_app = h5netcdf.File(out_nc, "a")

            # append index
            tvar = nc_app.variables[time_name]
            write_idx = int(tvar.shape[0])

            # resize variable first, write data
            v = nc_app.variables[varname]
            spatial_shape = v.shape[1:]
            v._h5ds.resize((write_idx + 1,) + spatial_shape)
            v[write_idx, ...] = coarse_da.values.astype(np.float32, copy=False)

            # then resize time, write time (commit)
            tvar._h5ds.resize((write_idx + 1,))
            tvar[write_idx] = s

            saved.add(s)
            appended += 1

    finally:
        if nc_app is not None:
            nc_app.close()

    print(
        f"\nUpdated NetCDF (one var): {out_nc}\n"
        f"  var: {varname}\n"
        f"  appended: {appended}\n"
        f"  skipped (already saved): {skipped_saved}\n"
        f"  skipped (missing hours): {skipped_missing}\n"
        f"  origin_iter={origin_iter}, step={step}, block_size={block_size}\n"
        f"  coarse box: box_j={box_j}, box_i={box_i}",
        flush=True,
    )


# -------------------------
# Example usage
# -------------------------
if __name__ == "__main__":
    path_scratch = "/scratch/jczhang/"
    path_MITgcm = "MITgcm_baroclinic_instability_old/MITgcm/verification/tutorial_reentrant_channel/run"
    grid_dir = path_scratch + path_MITgcm

    ds_fine = read_mitgcm.read_mitgrid_staggered(grid_dir)

    dims_map = {
        "U": ("k", "j", "i_g"),
        "V": ("k", "j_g", "i"),
        "W": ("k_l", "j", "i"),
        "T": ("k", "j", "i"),
        "S": ("k", "j", "i"),
    }
 # 5500080, 0013829040,  13901616
    start_time_index, end_time_index, step = 13901616, 26707984, 144
    box_j, box_i = 10, 10

    varname = "V"  # change to "U","V","W","S"

    load_T = make_mitgcm_loader_onevar(grid_dir, read_mitgcm, var=varname, dims=dims_map[varname])

    out_nc = f"./13901616_nc/{varname}_12hmean_coarse.nc"

    _12h_mean_to_netcdf_onevar(
        start_time_index=start_time_index,
        end_time_index=end_time_index,
        load_snapshot=load_T,
        out_nc=out_nc,
        ds_fine=ds_fine,
        box_j=box_j,
        box_i=box_i,
        varname=varname,
        block_size=12,
        step=step,
        time_name="time",
        skip_missing=False,
        print_every=1,
    )


[try] n=0/7410  block_start=13901616
[try] n=1/7410  block_start=13903344
[try] n=2/7410  block_start=13905072
[try] n=3/7410  block_start=13906800
[try] n=4/7410  block_start=13908528
[try] n=5/7410  block_start=13910256
[try] n=6/7410  block_start=13911984
[try] n=7/7410  block_start=13913712
[try] n=8/7410  block_start=13915440
[try] n=9/7410  block_start=13917168
[try] n=10/7410  block_start=13918896
[try] n=11/7410  block_start=13920624
[try] n=12/7410  block_start=13922352
[try] n=13/7410  block_start=13924080
[try] n=14/7410  block_start=13925808
[try] n=15/7410  block_start=13927536
[try] n=16/7410  block_start=13929264
[try] n=17/7410  block_start=13930992
[try] n=18/7410  block_start=13932720
[try] n=19/7410  block_start=13934448
[try] n=20/7410  block_start=13936176
[try] n=21/7410  block_start=13937904
[try] n=22/7410  block_start=13939632
[try] n=23/7410  block_start=13941360
[try] n=24/7410  block_start=13943088
[try] n=25/7410  block_start=13944816
[try] n=26/7410  block